In [1]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import ast
from typing import List
from sklearn.preprocessing import MultiLabelBinarizer

## I'm making a toy example to better understand what is happening when making your own custom transformer.

In [8]:
df = pd.DataFrame({'name': ['adam', 'adam', 'eve', 'adam', 'eve', 'eve'], 'val1': [1, 2, 3, 4, 5, 6], 'val2': [3, 4, 5, 6, 7, 8], 'target_val': [9, 8, 4, 6, 7, 7]})
df

,name,val1,val2,target_val
0,adam,1,3,9
1,adam,2,4,8
2,eve,3,5,4
3,adam,4,6,6
4,eve,5,7,7
5,eve,6,8,7


In [11]:
df.melt(id_vars="target_val", value_vars=['name']).groupby("value")['target_val'].mean()

value
adam    7.666667
eve     6.000000
Name: target_val, dtype: float64

In [19]:
class DirectorTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y):    # this is ONLY called during training!
        df = X.copy()
        df['_target'] = y

        self.global_mean = y.mean()

        stats = (
            df.melt(id_vars="_target", value_vars=["name"])
            .groupby("value")["_target"]
            .mean()
        )

        self.mapping_ = stats.to_dict()
        return self
    
    def transform(self, X): # this is called during training, validation and unseen data!

        results = []
        for d1 in X['name']:
            results.append(self.mapping_.get(d1, self.global_mean))
        return pd.DataFrame({'avg_personal_score': results}, index=X.index)
    
preprocess = ColumnTransformer(
    transformers=[
        ("name_of_person", DirectorTargetEncoder(), ["name"]),
    ],
    remainder="passthrough" # all other columns are automatically included
) 

In [ ]:
X_train1 = df.drop(columns='target_val')
y = df['target_val']
X_train2 = preprocess.fit_transform(X_train1, y)

# the output from prepocess is a numpy array. Irrelevant for ML algorithms, but can wrap in dataframe
# if I want to view it myself.
encoder_cols = ["avg_personal_score"]
passthrough_cols = [c for c in X_train1.columns if c != "name"]
all_cols = encoder_cols + passthrough_cols

X_train_df = pd.DataFrame(X_train2, columns=all_cols, index = X_train1.index)

In [25]:
X_train_df.head()

,avg_personal_score,val1,val2
0,7.666667,1.0,3.0
1,7.666667,2.0,4.0
2,6.000000,3.0,5.0
3,7.666667,4.0,6.0
4,6.000000,5.0,7.0
